# Práctica: Fine-Tuning en Azure AI Foundry

## ⚙️ Configuración del Entorno
Cargamos las credenciales y preparamos el cliente de Azure OpenAI.

In [ ]:
import os
import time
from dotenv import load_dotenv
from openai import AzureOpenAI

# Cargar variables desde el archivo .env en la raíz
load_dotenv(dotenv_path="../.env")

api_key = os.getenv("Key")
base_url = os.getenv("base_url")

if not api_key or not base_url:
    print("❌ Error: Asegúrate de tener configurado el archivo .env con 'Key' y 'base_url'")
else:
    # Limpiar el endpoint para el SDK de Azure
    azure_endpoint = base_url.split("/openai/v1")[0]
    
    client = AzureOpenAI(
        api_key=api_key,
        api_version="2024-05-01-preview",
        azure_endpoint=azure_endpoint
    )
    print(f"✅ Cliente configurado en: {azure_endpoint}")

## Sección 1: Introducción y Contexto

**Caso de uso:** Predicción del éxito académico de estudiantes basado en horas de estudio, asistencia y rendimiento previo.

**Dataset:** Dataset de 500 estudiantes (Kaggle) transformado a formato JSONL con roles de sistema, usuario y asistente.

## Sección 2: Preparación y Subida de Datos

In [ ]:
import os

# Nombres de los archivos
train_file = "training_set.jsonl"
val_file = "validation_set.jsonl"

if not os.path.exists(train_file) or not os.path.exists(val_file):
    print("⚠️ No se encontraron los archivos JSONL. Intentando generarlos...")
    if os.path.exists("prepare_datasets.py"):
        # Ejecutamos el script de preparación si existe
        %run prepare_datasets.py
    else:
        print("❌ Error: No se encuentra prepare_datasets.py. Por favor, asegúrate de que el script está en la carpeta.")
else:
    print("✅ Archivos de datos detectados y listos para subir.")

In [ ]:
def upload_file(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"No se encuentra el archivo: {file_path}")
    with open(file_path, "rb") as f:
        response = client.files.create(file=f, purpose="fine-tune")
    return response.id

try:
    training_id = upload_file(train_file)
    validation_id = upload_file(val_file)
    print(f"✅ Archivos subidos.\nTraining ID: {training_id}\nValidation ID: {validation_id}")
except Exception as e:
    print(f"❌ Error al subir archivos: {e}")

### 2.2 - Creación del Trabajo de Fine-Tuning

In [ ]:
base_model = "gpt-35-turbo-0613"  # Cambia esto por el modelo base permitido en tu región

try:
    job = client.fine_tuning.jobs.create(
        training_file=training_id,
        validation_file=validation_id,
        model=base_model
    )
    job_id = job.id
    print(f"🚀 Trabajo iniciado correctamente. ID: {job_id}")
except Exception as e:
    print(f"❌ Error al iniciar el trabajo: {e}")

## Sección 3: Monitorización

El proceso puede tardar un tiempo considerable. Esta celda monitoriza el progreso minuto a minuto.

In [ ]:
if 'job_id' in locals():
    while True:
        job_status = client.fine_tuning.jobs.retrieve(job_id)
        status = job_status.status
        print(f"[{time.strftime('%H:%M:%S')}] Estado del trabajo: {status}")
        
        if status in ["succeeded", "failed", "cancelled"]:
            break
        time.sleep(60)

    if status == "succeeded":
        print("🎉 ¡Fine-Tuning completado con éxito!")
        fine_tuned_model = job_status.fine_tuned_model
        print(f"Nombre del modelo resultante: {fine_tuned_model}")
        print("⚠️ Recuerda desplegar este modelo en Azure AI Foundry Studio para obtener su nombre de despliegue.")
else:
    print("❌ No hay un job_id activo. Ejecuta la celda anterior.")

## Sección 4: Pruebas Comparativas

Configura tus nombres de despliegue en el archivo `.env` para ejecutar esta sección.

In [ ]:
test_prompt = "Analiza mi perfil académico con estos datos: Horas de estudio: 12, Asistencia: 90%, Puntuación previa: 75, Actividades extra: Yes. ¿Cuál es mi predicción de éxito?"
system_prompt = "Eres un Asesor de Éxito Académico experto en análisis predictivo."

models = {
    "Base": os.getenv("BASE_MODEL_DEPLOYMENT"),
    "Fine-Tuned": os.getenv("FINETUNED_MODEL_DEPLOYMENT")
}

for name, model_name in models.items():
    if not model_name:
        print(f"⚠️ El despliegue de {name} no está configurado en el .env")
        continue
    
    try:
        print(f"--- Consultando Modelo {name} ({model_name}) ---")
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": test_prompt}
            ],
            max_tokens=60
        )
        print(f"Respuesta: {response.choices[0].message.content}")
    except Exception as e:
        print(f"❌ Error al consultar modelo {name}: {e}")
    print("\n")